In [ ]:
!pip -q install yfinance pandas numpy

import yfinance as yf
import pandas as pd
import numpy as np
import json
import time

from pathlib import Path
from datetime import datetime, timezone
from google.colab import files

print("yfinance:", yf.__version__)
print("pandas:", pd.__version__)

yfinance: 0.2.66
pandas: 2.2.3


In [ ]:
TICKERS = ["NVDA", "TSLA", "SPY"]
START = "2026-08-21"
END = "2026-10-03"

folder = Path("q2_data")
folder.mkdir(exist_ok=True)

all_stocks = []

for ticker in TICKERS:
    print(f"\nFetching {ticker}...")

    raw = yf.download(
        tickers=ticker,
        start=START,
        end=END,
        interval="1d",
        auto_adjust=False,
        back_adjust=False,
        repair=False,
        rounding=False,
        prepost=False,
        actions=False,
        threads=False,
        progress=False,
        multi_level_index=False,
        ignore_tz=False
    )

    if raw is None or raw.empty:
        raise RuntimeError(
            f"No data returned for {ticker}. Check the request."
        )

    # Preserve the original provider output.
    raw.to_csv(folder / f"{ticker}_raw.csv")

    # Use New York dates for the daily stock records.
    dates = pd.DatetimeIndex(raw.index)

    if dates.tz is not None:
        dates = dates.tz_convert("America/New_York")

    stock = raw[
        ["Open", "High", "Low", "Close", "Volume"]
    ].copy()

    stock.insert(0, "ticker", ticker)
    stock.insert(0, "date", dates.strftime("%Y-%m-%d"))

    all_stocks.append(stock.reset_index(drop=True))

    print(f"{ticker}: {len(stock)} records")

# Combine all three tickers.
df = pd.concat(all_stocks, ignore_index=True)

df = df.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

print("\nCombined dataset:")
print(df.head(10))


Fetching NVDA...
NVDA: 30 records

Fetching TSLA...
TSLA: 30 records

Fetching SPY...
SPY: 30 records

Combined dataset:
         date ticker        Open        High         Low       Close  \
0  2026-08-21   NVDA  218.419998  218.740005  214.500000  214.720001   
1  2026-08-24   NVDA  215.529999  215.589996  207.250000  208.479996   
2  2026-08-25   NVDA  211.029999  214.729996  210.110001  213.050003   
3  2026-08-26   NVDA  212.639999  213.600006  209.229996  209.660004   
4  2026-08-27   NVDA  222.860001  230.470001  220.899994  227.979996   
5  2026-08-28   NVDA  227.360001  229.259995  216.809998  217.550003   
6  2026-08-31   NVDA  218.869995  221.300003  216.210007  220.779999   
7  2026-09-01   NVDA  216.750000  220.410004  215.100006  217.440002   
8  2026-09-02   NVDA  218.789993  227.949997  218.479996  224.410004   
9  2026-09-03   NVDA  226.020004  230.399994  224.750000  228.449997   

      Volume  
0   98869300  
1  135187300  
2  122309000  
3  179900000  
4  2989098

In [ ]:
# Expected weekdays, excluding the September 7 market holiday.
expected_dates = set(
    pd.bdate_range(
        "2026-08-21",
        "2026-10-02"
    ).strftime("%Y-%m-%d")
) - {"2026-09-07"}

required = ["Open", "High", "Low", "Close", "Volume"]

assert len(expected_dates) == 30
assert len(df) == 90, f"Expected 90 rows, got {len(df)}"

assert not df.duplicated(["ticker", "date"]).any(), (
    "Duplicate ticker/date records found."
)

for ticker in TICKERS:
    dates = set(df.loc[df["ticker"] == ticker, "date"])
    missing = sorted(expected_dates - dates)
    extra = sorted(dates - expected_dates)

    print(
        ticker,
        "| Rows:", (df["ticker"] == ticker).sum(),
        "| Missing dates:", missing,
        "| Extra dates:", extra
    )

    assert not missing and not extra, (
        f"{ticker} has incorrect date coverage."
    )

assert df[required].notna().all().all(), (
    "Missing OHLC or Volume values."
)

assert np.isfinite(df[required].to_numpy(dtype=float)).all(), (
    "Non-finite numeric values found."
)

assert (df[["Open", "High", "Low", "Close"]] > 0).all().all()
assert (df["Volume"] >= 0).all()

assert (df["High"] >= df[["Open", "Low", "Close"]].max(axis=1)).all()
assert (df["Low"] <= df[["Open", "High", "Close"]].min(axis=1)).all()

print("\nAll data validation checks passed.")

NVDA | Rows: 30 | Missing dates: [] | Extra dates: []
TSLA | Rows: 30 | Missing dates: [] | Extra dates: []
SPY | Rows: 30 | Missing dates: [] | Extra dates: []

All data validation checks passed.


In [ ]:
df.to_csv(folder / "stock_daily.csv", index=False)

metadata = {
    "provider": "Yahoo Finance via yfinance",
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "yfinance_version": yf.__version__,
    "start": START,
    "end_exclusive": END,
    "interval": "1d",
    "tickers": TICKERS,
    "trading_timezone": "America/New_York",
    "auto_adjust": False,
    "back_adjust": False,
    "repair": False,
    "rounding": False,
    "prepost": False,
    "actions": False,
    "rows": len(df)
}

with open(folder / "provenance.json", "w") as f:
    json.dump(metadata, f, indent=2)

from google.colab import files
import shutil

shutil.make_archive("q2_data", "zip", folder)
files.download("q2_data.zip")

print("Saved the validated data and metadata.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved the validated data and metadata.


In [ ]:
# Work with the validated dataset from the previous step.
df = df.sort_values(["ticker", "date"]).copy()

# Get the August 21 closing price for each ticker.
first_close = df.groupby("ticker")["Close"].transform("first")

# Normalized closing-price index.
df["normalized_index"] = 100 * df["Close"] / first_close

# Daily price swing, as a percentage.
df["daily_swing_pct"] = (
    100 * (df["High"] - df["Low"]) / df["Open"]
)

# Alert thresholds: 3% and 5%.
df["alert_3pct"] = df["daily_swing_pct"] >= 3
df["alert_5pct"] = df["daily_swing_pct"] >= 5

# Summarize each ticker's 30 trading days.
summary = df.groupby("ticker").agg(
    trading_days=("date", "count"),
    first_close=("Close", "first"),
    last_close=("Close", "last"),
    avg_daily_swing_pct=("daily_swing_pct", "mean"),
    alerts_3pct=("alert_3pct", "sum"),
    alerts_5pct=("alert_5pct", "sum")
).reset_index()

summary["window_return_pct"] = (
    100
    * (summary["last_close"] - summary["first_close"])
    / summary["first_close"]
)

# Round only for display, not in the saved calculations.
display(summary.round(2))

,ticker,trading_days,first_close,last_close,avg_daily_swing_pct,alerts_3pct,alerts_5pct,window_return_pct
0,NVDA,30,214.72,233.95,2.37,6,1,8.96
1,SPY,30,765.72,769.64,0.67,0,0,0.51
2,TSLA,30,362.86,370.59,3.17,11,2,2.13


In [ ]:
workload_rows = []
alert_rows = []

for threshold in [3, 5]:

    # Find all alerts at this threshold.
    alerts = df[df["daily_swing_pct"] >= threshold].copy()

    # Examine each of the 30 trading dates.
    for date in sorted(df["date"].unique()):

        daily_alerts = alerts[
            alerts["date"] == date
        ].sort_values(
            ["daily_swing_pct", "ticker"],
            ascending=[False, True]
        )

        count = len(daily_alerts)

        # Select at most one asset for manual review.
        selected = (
            daily_alerts.iloc[0]["ticker"]
            if count > 0
            else None
        )

        workload_rows.append({
            "date": date,
            "threshold_pct": threshold,
            "asset_alerts": count,
            "selected_asset": selected,
            "unreviewed_alerts": max(count - 1, 0)
        })

        for _, row in daily_alerts.iterrows():
            alert_rows.append({
                "date": date,
                "threshold_pct": threshold,
                "ticker": row["ticker"],
                "daily_swing_pct": row["daily_swing_pct"],
                "selected_for_review": row["ticker"] == selected
            })

workload = pd.DataFrame(workload_rows)
alert_details = pd.DataFrame(alert_rows)

workload_summary = workload.groupby("threshold_pct").agg(
    total_asset_alerts=("asset_alerts", "sum"),
    dates_with_alerts=("asset_alerts", lambda x: (x > 0).sum()),
    simultaneous_alert_dates=("asset_alerts", lambda x: (x > 1).sum()),
    unreviewed_alerts=("unreviewed_alerts", "sum")
).reset_index()

display(workload_summary)

,threshold_pct,total_asset_alerts,dates_with_alerts,simultaneous_alert_dates,unreviewed_alerts
0,3,17,14,3,3
1,5,3,3,0,0


In [ ]:
df.to_csv(
    folder / "stock_metrics.csv",
    index=False
)

summary.to_csv(
    folder / "stock_summary.csv",
    index=False
)

workload.to_csv(
    folder / "daily_workload.csv",
    index=False
)

workload_summary.to_csv(
    folder / "workload_summary.csv",
    index=False
)

alert_details.to_csv(
    folder / "alert_details.csv",
    index=False
)

print("All Q2 calculations saved.")

display(summary.round(2))
display(workload_summary)

All Q2 calculations saved.


,ticker,trading_days,first_close,last_close,avg_daily_swing_pct,alerts_3pct,alerts_5pct,window_return_pct
0,NVDA,30,214.72,233.95,2.37,6,1,8.96
1,SPY,30,765.72,769.64,0.67,0,0,0.51
2,TSLA,30,362.86,370.59,3.17,11,2,2.13


,threshold_pct,total_asset_alerts,dates_with_alerts,simultaneous_alert_dates,unreviewed_alerts
0,3,17,14,3,3
1,5,3,3,0,0


In [ ]:
import shutil
from google.colab import files

shutil.make_archive(
    "q2_complete_data",
    "zip",
    root_dir=".",
    base_dir="q2_data"
)

files.download("q2_complete_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("=== 1. STOCK COMPARISON ===")

display(
    summary[[
        "ticker",
        "window_return_pct",
        "avg_daily_swing_pct",
        "alerts_3pct",
        "alerts_5pct"
    ]].round(2)
)

print("\n=== 2. MONITORING WORKLOAD ===")

display(workload_summary)

print("\n=== 3. DATES WITH SIMULTANEOUS ALERTS ===")

for threshold in [3, 5]:
    print(f"\nThreshold: {threshold}%")

    simultaneous = workload[
        (workload["threshold_pct"] == threshold) &
        (workload["asset_alerts"] > 1)
    ]

    display(
        simultaneous[[
            "date",
            "asset_alerts",
            "selected_asset",
            "unreviewed_alerts"
        ]]
    )

print("\n=== 4. TOP 5 LARGEST DAILY SWINGS ===")

display(
    df.nlargest(5, "daily_swing_pct")[[
        "date",
        "ticker",
        "daily_swing_pct"
    ]].round(2)
)

=== 1. STOCK COMPARISON ===


,ticker,window_return_pct,avg_daily_swing_pct,alerts_3pct,alerts_5pct
0,NVDA,8.96,2.37,6,1
1,SPY,0.51,0.67,0,0
2,TSLA,2.13,3.17,11,2



=== 2. MONITORING WORKLOAD ===


,threshold_pct,total_asset_alerts,dates_with_alerts,simultaneous_alert_dates,unreviewed_alerts
0,3,17,14,3,3
1,5,3,3,0,0



=== 3. DATES WITH SIMULTANEOUS ALERTS ===

Threshold: 3%


,date,asset_alerts,selected_asset,unreviewed_alerts
1,2026-08-24,2,TSLA,1
5,2026-08-28,2,NVDA,1
11,2026-09-08,2,TSLA,1



Threshold: 5%


,date,asset_alerts,selected_asset,unreviewed_alerts



=== 4. TOP 5 LARGEST DAILY SWINGS ===


,date,ticker,daily_swing_pct
66,2026-08-31,TSLA,6.27
60,2026-08-21,TSLA,5.60
5,2026-08-28,NVDA,5.48
84,2026-09-25,TSLA,4.98
69,2026-09-03,TSLA,4.98
